# Insurance Pricing & Profitability Analysis

## Context
This notebook serves as the analytical core of the **Pricing & Profitability Agent**. Its primary objective is to evaluate the financial performance of an insurance portfolio by analyzing the relationship between the premiums collected, the claims paid out, and the expenses incurred.

## Objectives
- Calculate fundamental insurance metrics: **Loss Ratio**, **Expense Ratio**, **Combined Ratio**, and **Underwriting Profit**.
- Segment profitability across different **Product Types**, **Customer Segments**, **Distribution Channels**, and **States**.
- Track monthly profit trends to identify seasonal or temporal patterns.
- Provide automated pricing insights to identify underpriced (loss-making) or highly profitable products.

## Concepts Used
- **Loss Ratio**: $\frac{\text{Claims Paid}}{\text{Premiums Earned}}$. Indicates the percentage of premium used to pay claims. A ratio > 1.0 means the company is paying out more in claims than it earns in premiums.
- **Expense Ratio**: $\frac{\text{Expenses}}{\text{Premiums Earned}}$. Indicates the efficiency of operations.
- **Combined Ratio**: $\text{Loss Ratio} + \text{Expense Ratio}$. The ultimate measure of underwriting profitability. A ratio < 1.0 indicates an underwriting profit, while > 1.0 indicates a loss.
- **Underwriting Profit**: $\text{Premiums} - \text{Claims} - \text{Expenses}$. The absolute monetary profit or loss from core insurance operations.

## 1. Import Dependencies
We use `pandas` for data manipulation, `numpy` for numerical operations, and `matplotlib`/`seaborn` for generating visual insights (replacing the Streamlit UI components from the original script).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Notebook visual settings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Data Loading
Load the insurance dataset. We will use the `Cleaned_Data` from the validation report.

In [ ]:
# Try loading the cleaned excel file. Update path as necessary.
try:
    df = pd.read_excel('reports/validation_report.xlsx', sheet_name='Cleaned_Data')
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print("Dataset not found. Please provide a valid file path.")
    # Example fallback: df = pd.read_csv('your_data.csv')

## 3. Data Preprocessing
Before calculating metrics, we must ensure data integrity:
1. **Date Formatting**: Convert the `Date` column to datetime objects.
2. **Numeric Conversion**: Ensure financial columns (`Written_Premium`, `Claim_Amount`, `Total_Expense`) are numeric to prevent calculation errors.
3. **Missing Values**: Drop rows without `Written_Premium` since it's the denominator for our core ratios. Handle zero-premiums to avoid division by zero.

In [ ]:
if 'df' in locals():
    # 1. Convert Date
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # 2. Numeric Conversion
    numeric_cols = ["Written_Premium", "Claim_Amount", "Total_Expense"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 3. Handle Missing/Zero Premium
    # Drop rows where Premium is NaN
    df = df.dropna(subset=["Written_Premium"])
    if 'Written_Premium' in df.columns:
        df = df.dropna(subset=['Written_Premium'])
        premium_safe = df['Written_Premium'].replace(0, np.nan)
    else:
        premium_safe = np.nan


## 4. Core Financial Metrics Calculation
Here we compute the foundational KPIs of insurance pricing:
- `Loss_Ratio` = Claim Amount / Written Premium
- `Expense_Ratio` = Total Expense / Written Premium
- `Combined_Ratio` = Loss Ratio + Expense Ratio
- `Underwriting_Profit` = Written Premium - Claim Amount - Total Expense

In [ ]:
if 'df' in locals():
    # Ratio Calculations
    df["Loss_Ratio"] = df["Claim_Amount"] / premium_safe
    df["Expense_Ratio"] = df["Total_Expense"] / premium_safe
    df["Combined_Ratio"] = df["Loss_Ratio"] + df["Expense_Ratio"]

    # Absolute Profit Calculation
    df["Underwriting_Profit"] = df["Written_Premium"] - df["Claim_Amount"] - df["Total_Expense"]
    
    display(df[["Written_Premium", "Claim_Amount", "Total_Expense", "Loss_Ratio", "Combined_Ratio", "Underwriting_Profit"]].head())

## 5. Profitability Classification
To make the data actionable, we classify policies into Profitability Tiers based on their Combined Ratio:
- **Excellent**: < 80% (High margins)
- **Good**: 80% - 94.9% (Healthy margins)
- **Marginal**: 95% - 100% (Barely breaking even)
- **Loss-Making**: > 100% (Losing money on underwriting)

In [ ]:
def classify_ratio(x):
    if pd.isna(x):
        return "Unknown"
    if x < 0.80:
        return "Excellent"
    elif x < 0.95:
        return "Good"
    elif x <= 1.00:
        return "Marginal"
    else:
        return "Loss-Making"

if 'df' in locals():
    df["Profitability_Tier"] = df["Combined_Ratio"].apply(classify_ratio)
    print("Profitability classification applied.")

## 6. Portfolio Overview & Key Performance Indicators (KPIs)
A macro-level view of the entire portfolio's performance.

In [ ]:
if 'df' in locals():
    total_premium = df['Written_Premium'].sum()
    total_claims = df['Claim_Amount'].sum()
    total_expenses = df['Total_Expense'].sum()
    total_profit = df['Underwriting_Profit'].sum()

    print("="*40)
    print("          PORTFOLIO OVERVIEW")
    print("="*40)
    print(f"Total Premium:      â‚¹ {total_premium:,.0f}")
    print(f"Total Claims:       â‚¹ {total_claims:,.0f}")
    print(f"Total Expenses:     â‚¹ {total_expenses:,.0f}")
    print(f"Underwriting Profit:â‚¹ {total_profit:,.0f}")
    print("="*40)

## 7. Product-Level Analysis
Understanding which products drive profitability and which are detrimental.
- We calculate the **average Loss Ratio** per product to identify risk.
- We calculate the **total Underwriting Profit** per product to see monetary impact.

In [ ]:
if 'df' in locals() and "Product_Type" in df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Loss Ratio by Product
    loss_ratio_product = df.groupby("Product_Type")["Loss_Ratio"].mean().sort_values(ascending=False)
    sns.barplot(x=loss_ratio_product.values, y=loss_ratio_product.index, ax=ax1, palette="Reds_r")
    ax1.set_title("Average Loss Ratio by Product")
    ax1.set_xlabel("Loss Ratio")
    ax1.axvline(1.0, color='red', linestyle='--') # Mark the 1.0 threshold
    
    # Profit by Product
    product_profit = df.groupby("Product_Type")["Underwriting_Profit"].sum().sort_values(ascending=False)
    sns.barplot(x=product_profit.values, y=product_profit.index, ax=ax2, palette="Greens_r")
    ax2.set_title("Total Underwriting Profit by Product")
    ax2.set_xlabel("Profit (â‚¹)")
    
    plt.tight_layout()
    plt.show()

## 8. Distribution by Profitability Tiers
Visualizing the volume of policies falling into each tier.

In [ ]:
if 'df' in locals():
    tier_summary = df.groupby("Profitability_Tier").size().reset_index(name="Count")
    display(tier_summary)
    
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x="Profitability_Tier", order=["Excellent", "Good", "Marginal", "Loss-Making", "Unknown"], palette="viridis")
    plt.title("Volume of Policies by Profitability Tier")
    plt.ylabel("Number of Policies")
    plt.show()

## 9. Segment, Channel, and State Profitability
Slicing the Underwriting Profit across different dimensions: Customer Segment, Distribution Channel, and Geographic State.

In [ ]:
if 'df' in locals():
    # Customer Segment
    if "Customer_Segment" in df.columns:
        segment_profit = df.groupby("Customer_Segment")["Underwriting_Profit"].sum().sort_values(ascending=False)
        print("\nProfit by Customer Segment:")
        display(segment_profit.reset_index())
    
    # Distribution Channel
    if "Distribution_Channel" in df.columns:
        channel_profit = df.groupby("Distribution_Channel")["Underwriting_Profit"].sum().sort_values(ascending=False)
        print("\nProfit by Distribution Channel:")
        display(channel_profit.reset_index())
        
    # State-wise Profitability
    if "State" in df.columns:
        state_profit = df.groupby("State")["Underwriting_Profit"].sum().sort_values(ascending=False).head(10) # Top 10
        plt.figure(figsize=(12, 5))
        sns.barplot(x=state_profit.index, y=state_profit.values, palette="Blues_r")
        plt.title("Top 10 States by Underwriting Profit")
        plt.xticks(rotation=45)
        plt.ylabel("Profit (â‚¹)")
        plt.show()

## 10. Monthly Profit Trend (Time Series Analysis)
Aggregating profitability over time to detect seasonal patterns or long-term trends. We group by Month End (`ME`) using `pd.Grouper`.

In [ ]:
if 'df' in locals() and "Date" in df.columns:
    monthly_profit = (
        df.groupby(pd.Grouper(key="Date", freq="ME"))["Underwriting_Profit"]
        .sum()
        .tail(36) # Last 36 months
    )
    
    plt.figure(figsize=(14, 6))
    sns.lineplot(x=monthly_profit.index, y=monthly_profit.values, marker="o", color="b")
    plt.title("Monthly Underwriting Profit Trend (Last 36 Months)")
    plt.xlabel("Date")
    plt.ylabel("Total Profit (â‚¹)")
    plt.axhline(0, color='red', linestyle='--') # Break-even line
    plt.grid(True)
    plt.show()

## 11. AI Pricing Insights
Automated heuristic analysis to provide strategic recommendations based on the Combined Ratio of each product line.
- **Ratio > 1.0**: Underpriced / Loss-Making. Immediate rate action required.
- **Ratio < 0.8**: Highly Profitable. Indicates pricing power; potential for growth without sacrificing margins.
- **0.8 <= Ratio <= 1.0**: Monitor Pricing. Margins are tight.

In [ ]:
if 'df' in locals() and "Product_Type" in df.columns:
    print("=== AI PRICING INSIGHTS ===\n")
    
    ratio_table = df.groupby("Product_Type")["Combined_Ratio"].mean().reset_index()
    
    for _, row in ratio_table.iterrows():
        product = row["Product_Type"]
        ratio = row["Combined_Ratio"]
        
        if ratio > 1:
            print(f"ðŸ”´ {product}: Combined Ratio {ratio:.2f} â†’ Underpriced / Loss-Making")
        elif ratio < 0.80:
            print(f"ðŸŸ¢ {product}: Combined Ratio {ratio:.2f} â†’ Highly Profitable")
        else:
            print(f"ðŸŸ¡ {product}: Combined Ratio {ratio:.2f} â†’ Monitor Pricing")